## 2026_Winter_DL_week5_Homework
CNN 실습 예제코드입니다!

이번 5주차 숙제는 총 2가지로 구성되어 있습니다.
- 논문 리뷰 - ResNet : https://arxiv.org/abs/1512.03385
    - 다음 논문을 읽고 팀별로 논문 스터디 및 정리를 준비해주시면 됩니다! (notion 이나 ppt 로 논문 정리 자료 제작 후에 팀별로 git에 업로드 해주세요 :D)

- 코드 과제 (아래 두 가지 중 택1!)
    - 1) (CNN 을 처음 접해보시거나 기초를 더 다지고 싶은 분들께 추천)
        - 아래 실습코드를 그대로 참고하셔도 좋고, 새롭게 짜셔도 좋습니다. 데이터셋은 CIFAR10이 아닌 벤치마크(MNIST, FFHQ, ImageNet)이나 다른 데이터셋을 사용하되, 학습 시간 및 리소스를 적절히 사용할 수 있는 화질을 사용하는 것을 추천 드립니다. 또한, 앞서 배운 regularization, initialization, optimizer 등등 기법을 추가해보시거나, layer를 변형하는 시도를 추가하여 결과를 분석해주세요.
        - example : Earlystopping 추가, Dropoutlayer 추가, batch nomalization 추가, stride 및 padding 변형, Conv layer 추가 및 삭제 등등
    - 2) (이미 CNN 지식이 있는 분들)
        - ResNet 을 논문 "**만**" 읽고 핵심 모듈 코드를 직접 짜보세요! 코드를 보지 않고 conv block이나 layer들을 논문 기반으로 짜보시면 됩니다.(임교수님께서 추천해주신 공부법입니다 ㅎㅎ)

# RestNet 핵심 모듈 구현

In [ ]:
# ResNet Model
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, downsample=None):
        super().__init__()

        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.downsample = downsample

    def forward(self, x):
        ident = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        if self.downsample is not None:
            ident = self.downsample(x)

        out += ident
        out = self.relu(out)
        return out

# 내가 짠 코드 분석 + 핵심 설명
## 1. layer1에서 downsample이 생성되는가?

**생성되지 않는다.**

```python
if stride != 1 or self.in_planes != planes * BasicBlock.expansion:
```
`layer1`을 만들 때 `stride=1`, `self.in_planes=64`, `planes=64`, `expansion=1`이니:
- `stride != 1` → `1 != 1` → **False**
- `self.in_planes != planes*expansion` → `64 != 64*1` → **False**

`False or False` → downsample = `None`.

**왜 이게 맞는가:** stem을 통과한 직후 채널이 이미 64인데, `layer1`도 채널을 64로 유지하고(stride=1이라 공간 크기도 그대로) 그대로 이어받으니, ident(입력 x)와 `conv1→conv2`를 거친 출력의 **shape이 처음부터 똑같다.** shape이 같으니 `out += ident`가 아무 변환 없이 바로 더해질 수 있고, 굳이 1×1 conv로 차원을 맞춰줄 필요가 없는 것이다. `layer2`부터는 채널도(64→128), 공간 크기도(stride=2로 절반) 바뀌니 downsample이 필요해진다.

## 2. 왜 `bias=False`인가?

**바로 다음에 오는 `BatchNorm2d` 때문이다.**

BatchNorm의 계산식을 떠올려보면:
$$y = \gamma\hat{x} + \beta$$
BN 자체가 이미 학습 가능한 이동 파라미터 $\beta$를 갖고 있다. 만약 conv에 편향 $b$까지 있다면:
$$z = Wx + b \;\;\rightarrow\;\; \hat{z} = \frac{(Wx+b) - \mu}{\sigma} \;\;\rightarrow\;\; y = \gamma\hat{z}+\beta$$
여기서 $b$가 정규화 과정(평균을 빼는 단계)에서 **사실상 상쇄되어 사라진다** — $b$가 모든 샘플에 똑같이 더해지는 상수라서, 배치 평균 $\mu$를 계산할 때 그 $b$도 같이 평균에 포함되어 빼는 순간 없어져 버린다. 즉 **conv의 `bias`는 BN 뒤에서는 아무 역할도 못 하는 낭비되는 파라미터**라서, 아예 꺼두는 게 표준 관행이다.

## 3. 왜 첫 번째 블록만 다르게 처리하는가?

```python
layers.append(BasicBlock(self.in_planes, planes, stride, downsample))  # 첫 블록
self.in_planes = planes * BasicBlock.expansion
for _ in range(1, blocks):
    layers.append(BasicBlock(self.in_planes, planes))  # 나머지 블록들
```

채널 수와 공간 크기가 바뀌는 지점은 딱 한 번, stage의 맨 처음뿐이기 때문이다.

예를 들어 `layer2`(planes=128, stride=2, blocks=2)를 보면:
- **첫 번째 블록**: 입력이 아직 `layer1`에서 넘어온 64채널이다. 그러니 `64→128`로 채널을 늘리고, `stride=2`로 공간도 줄여야 한다 — 그래서 `downsample`(1×1 conv로 ident도 같이 64→128, stride=2로 변환)이 필요하다.
- **두 번째 블록**: 입력이 이미 첫 번째 블록을 거쳐 128채널이 된 상태다. 여기서부턴 그냥 "128채널을 유지한 채 특징만 더 다듬는" 역할이라, `stride=1`(공간 크기 유지)이고 채널도 그대로라 `downsample`이 필요 없다.

**비유하자면:** 첫 블록이 "새 층(stage)으로 넘어가는 관문" 역할을 하고, 나머지 블록들은 그 층 안에서 "같은 크기로 계속 특징을 정제하는" 역할을 한다. 관문에서만 크기 변환이 필요하니, 그 지점에서만 `stride`와 `downsample`을 다르게 주는 것이다.


`MyResNet` 클래스(stem + `_make_layer` + `layer1~4` + `avgpool` + `fc`)는 그 `BasicBlock`을 여러 개 쌓아서 **ResNet-18이라는 완성된 전체 네트워크**로 조립한 부분이다.